In [1]:
import math
import warnings
from pathlib import Path
from plotly import graph_objects as go
import numpy as np
import numpy.typing as npt
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import wandb
import random
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import DataLoader, Dataset

warnings.filterwarnings("ignore")

In [2]:
SEED = 4242676968

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Makes cuDNN deterministic (slight perf cost, irrelevant on MPS)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [3]:
DATA_DIR = Path("../data")
CHECKPOINT_DIR = Path("./checkpoints")
CHECKPOINT_DIR.mkdir(exist_ok=True)
DEVICE = (
    torch.device("cuda") if torch.cuda.is_available() else 
    torch.device("mps") if torch.backends.mps.is_available() else
    torch.device("cpu")
)

## Data Loading

In [4]:
class WindowDataset(Dataset):
    def __init__(
        self,
        df: pd.DataFrame,
        feature_cols: list[str],
        seq_len: int = 20,
        target_col: str = "target",
    ):
        self.seq_len = seq_len
        self.samples: list[tuple[npt.NDArray[np.float32], np.float32]] = []

        for ticker, group in df.groupby("ticker"):
            group = group.sort_index()
            X = group[feature_cols].values.astype(np.float32)
            y = group[target_col].values.astype(np.float32)

            mask = np.isfinite(X).all(axis=1) & np.isfinite(y)
            X = X[mask]
            y = y[mask]

            for i in range(seq_len, len(group)):
                window = X[i - seq_len : i]
                target = y[i]
                self.samples.append((window, target))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        x, y = self.samples[idx]
        return torch.from_numpy(x), torch.tensor(y)

In [5]:
def make_loaders(
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    test_df: pd.DataFrame,
    feature_cols: list[str],
    seq_len: int = 20,
    batch_size: int = 64,
) -> tuple[DataLoader, DataLoader, DataLoader]:
    kw = dict(feature_cols=feature_cols, seq_len=seq_len)
    train_ds = WindowDataset(train_df, **kw)
    val_ds = WindowDataset(val_df, **kw)
    test_ds = WindowDataset(test_df, **kw)

    loader_kw = dict(batch_size=batch_size, num_workers=0)  # num_workers=0 on MPS
    return (
        DataLoader(train_ds, shuffle=True, **loader_kw),
        DataLoader(val_ds, shuffle=False, **loader_kw),
        DataLoader(test_ds, shuffle=False, **loader_kw),
    )

## Model

In [6]:
class DirectionalHuberLoss(nn.Module):
    """Huber + bonus when sign(pred) == sign(target)."""
    def __init__(self, delta=1.0, dir_weight=0.3):
        super().__init__()
        self.huber = nn.HuberLoss(delta=delta, reduction="none")
        self.dir_weight = dir_weight

    def forward(self, preds, targets):
        huber = self.huber(preds, targets).mean()
        # Reward correct direction, penalize wrong direction
        sign_match = (preds.sign() == targets.sign()).float()
        dir_loss = 1.0 - sign_match.mean()
        return huber + self.dir_weight * dir_loss

In [7]:
class AdditiveAttention(nn.Module):
    """
    Bahdanau-style attention over the time dimension.

    Given hidden states H  (B, T, H), produces a context vector (B, H)
    that is a weighted sum of all timesteps.

    Score:  e_t = v · tanh(W · h_t)      (learned scalar per timestep)
    Weight: α   = softmax(e)              (over T)
    Output: c   = Σ α_t * h_t
    """

    def __init__(self, hidden_size: int):
        super().__init__()
        self.W = nn.Linear(hidden_size, hidden_size, bias=True)
        self.v = nn.Linear(hidden_size, 1, bias=False)

    def forward(self, hidden_states: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        # hidden_states: (B, T, H)
        scores = self.v(torch.tanh(self.W(hidden_states)))  # (B, T, 1)
        weights = torch.softmax(scores, dim=1)  # (B, T, 1)
        context = (weights * hidden_states).sum(dim=1)  # (B, H)
        return context, weights.squeeze(-1)  # context, attn_map

In [8]:
class DirectionalBCELoss(nn.Module):
    """Treat return prediction as a classification: positive vs negative."""
    def forward(self, preds, targets):
        # Convert targets to binary labels: 1 if positive, 0 if negative
        labels = (targets > 0).float()
        # Sigmoid on raw predictions → probability of "up"
        return F.binary_cross_entropy_with_logits(preds, labels)

In [9]:
class SoftDirectionalHuberLoss(nn.Module):
    """Huber + differentiable directional penalty."""

    def __init__(self, delta=1.01, dir_weight=0.3, sharpness=10.0):
        super().__init__()
        self.huber = nn.HuberLoss(delta=delta, reduction="none")
        self.dir_weight = dir_weight
        self.sharpness = sharpness  # higher = closer to hard sign

    def forward(self, preds, targets):
        huber = self.huber(preds, targets).mean()

        # Approach 1: soft sign agreement via tanh
        # Product is positive when same sign, negative when opposite
        agreement = torch.tanh(preds * self.sharpness) * torch.tanh(targets * self.sharpness)
        dir_loss = -agreement.mean()  # minimize → maximize agreement

        return huber + self.dir_weight * dir_loss



In [10]:
class GRUWithAttention(nn.Module):
    def __init__(
        self,
        input_size: int,
        hidden_size: int,
        num_layers: int = 1,
        linear_hidden: int = 64,
        dropout: float = 0.2,
        skip_dropout: float = 0.3,
        skip_days: int = 10,        # ← NEW: how many recent timesteps to bypass
    ):
        super().__init__()
        self.skip_days = skip_days

        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.attention = AdditiveAttention(hidden_size)

        # Skip path: flatten last skip_days of raw features → project to hidden_size
        self.skip = nn.Sequential(
            nn.Linear(skip_days * input_size, hidden_size),
            nn.GELU(),
            nn.Dropout(skip_dropout),
        )

        # Head now sees attention_context (H) + skip (H) = 2H
        self.head = nn.Sequential(
            nn.Linear(2 * hidden_size, linear_hidden),
            nn.GELU(),
            nn.LayerNorm(linear_hidden),
            nn.Dropout(dropout),
            nn.Linear(linear_hidden, 1),
        )

    def forward(self, x: torch.Tensor):
        # x: (B, T, F)

        # Skip path: raw recent features, bypassing GRU
        skip = self.skip(x[:, -self.skip_days:].reshape(x.size(0), -1))  # (B, H)

        # GRU path: full sequence temporal processing
        hidden_states, _ = self.gru(x)          # (B, T, H)
        context, attn = self.attention(hidden_states)  # (B, H)

        # Concatenate both views for the head
        combined = torch.cat([context, skip], dim=1)  # (B, 2H)
        out = self.head(combined).squeeze(-1)          # (B,)
        return out, attn

## Model torchure

In [11]:
def train_one_epoch(
    model: GRUWithAttention,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    criterion: nn.Module,
    device: torch.device,
    clip_grad: float = 1.0,
) -> float:
    model.train()
    total_loss = 0.0

    for x, y in loader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        preds, _ = model(x)
        loss = criterion(preds, y)
        loss.backward()

        if clip_grad > 0:
            nn.utils.clip_grad_norm_(model.parameters(), clip_grad)

        optimizer.step()
        total_loss += loss.item() * len(y)

    return total_loss / len(loader.dataset)

In [12]:
@torch.no_grad()
def evaluate(
    model: GRUWithAttention,
    loader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
) -> dict:
    """
    Returns a dict with loss, MAE, and directional accuracy.

    Directional accuracy: fraction of predictions where sign(pred) == sign(true).
    This is the headline metric for a return-prediction model –
    a coin-flip baseline is 0.50.
    """
    model.eval()
    all_preds, all_targets = [], []

    for x, y in loader:
        x, y = x.to(device), y.to(device)
        preds, _ = model(x)
        all_preds.append(preds.cpu())
        all_targets.append(y.cpu())

    preds = torch.cat(all_preds)
    targets = torch.cat(all_targets)

    loss = criterion(preds, targets).item()
    mae = (preds - targets).abs().mean().item()
    dir_acc = (preds.sign() == targets.sign()).float().mean().item()

    return {"loss": loss, "mae": mae, "dir_acc": dir_acc}

In [13]:
def train(
    model:        GRUWithAttention,
    train_loader: DataLoader,
    val_loader:   DataLoader,
    device:       torch.device,
    lr:           float = 1e-4,
    epochs:       int   = 50,
    patience:     int   = 10,
    huber_delta:  float = 1.0,
) ->GRUWithAttention:
    """
    Full training loop with:
     - Huber loss (delta=1.0 on z-scored targets is well calibrated)
     - AdamW + cosine LR schedule
     - Early stopping on val loss
     - W&B logging
    """
    criterion = SoftDirectionalHuberLoss(delta=huber_delta)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    best_val_loss = float("inf")
    patience_ctr  = 0
    best_state    = None

    logs = []

    for epoch in range(1, epochs + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
        val_metrics = evaluate(model, val_loader, criterion, device)
        scheduler.step()

        logs.append({
            "epoch":          epoch,
            "train/loss":     train_loss,
            "val/loss":       val_metrics["loss"],
            "val/mae":        val_metrics["mae"],
            "val/dir_acc":    val_metrics["dir_acc"],
                "lr":             scheduler.get_last_lr()[0],
            })

        print(
            f"Epoch {epoch:3d} | "
            f"train_loss={train_loss:.4f}  "
            f"val_loss={val_metrics['loss']:.4f}  "
            f"val_dir_acc={val_metrics['dir_acc']:.3f}"
        )

        # Early stopping
        if val_metrics["loss"] < best_val_loss:
            best_val_loss = val_metrics["loss"]
            best_state    = {k: v.clone() for k, v in model.state_dict().items()}
            patience_ctr  = 0
        else:
            patience_ctr += 1
            if patience_ctr >= patience:
                print(f"Early stop at epoch {epoch}.")
                break

    model.load_state_dict(best_state)
    return model


In [14]:
train_df = pd.read_csv("data/train_features.csv", index_col=0, parse_dates=True)
val_df   = pd.read_csv("data/val_features.csv",   index_col=0, parse_dates=True)
test_df  = pd.read_csv("data/test_features.csv",  index_col=0, parse_dates=True)


# Combine train + val for CV; keep test held out as final evaluation
cv_df   = pd.concat([train_df, val_df]).sort_index()
cv_df.sort_index(inplace=True)
print(f"CV pool:  {len(cv_df):,} rows  |  "
      f"{cv_df.index.min().date()} → {cv_df.index.max().date()}")
print(f"Test set: {len(test_df):,} rows (held out)")

feature_cols = [
    # Regime / trend (strongest single predictor, r=-0.067)
    "bull_regime",
    
    # Momentum (r=+0.045, captures "where in Bollinger Band")
    "z_bb_pct_b",
    
    # Volume anomaly (r=+0.041, orthogonal to price features)
    "z_volume_z20",
    
    # Yesterday's return (r=-0.021, short-term mean reversion signal)
    "z_log_close_return_1",
    
    # Volatility (r=+0.031, "how wild was today")
    "z_range",
    
    # Autocorrelation (r=+0.024, trending vs mean-reverting regime)
    "z_ret_autocorr",
    
    # Calendar (weak but orthogonal to everything else)
    "dow_cos",
    
    # Vol regime (binary, orthogonal to z_range which measures magnitude)
    "high_vol_regime",
]
print(f"Features ({len(feature_cols)}): {feature_cols[:5]} ...")

CV pool:  12,011 rows  |  2019-07-18 → 2025-07-09
Test set: 1,464 rows (held out)
Features (8): ['bull_regime', 'z_bb_pct_b', 'z_volume_z20', 'z_log_close_return_1', 'z_range'] ...


In [30]:
from timeseries_cv import cross_validate, summarize_cv, compare_strategies
from timeseries_cv import expanding_window_folds, sliding_window_folds

criterion = SoftDirectionalHuberLoss(delta=1.01, dir_weight=0.3, sharpness=10.0) 

print("EXPANDING:")
for f in expanding_window_folds(cv_df, n_folds=5, min_train_frac=0.40, val_frac=0.10):
    print(f"  Fold {f.fold_idx}: train {f.train_start.date()} → {f.train_end.date()} "
          f"| val {f.val_start.date()} → {f.val_end.date()}")
    targets  = cv_df.loc[f.val_start : f.val_end, "target"]
    loss = criterion(
        torch.zeros_like(torch.from_numpy(targets.values)),  # dummy preds
        torch.from_numpy(targets.values)
    ).item()
    print(f"    Huber loss on val set (dummy preds): {loss:.4f}")

print("\nSLIDING:")
for f in sliding_window_folds(cv_df, n_folds=5, train_frac=0.40, val_frac=0.10):
    print(f"  Fold {f.fold_idx}: train {f.train_start.date()} → {f.train_end.date()} "
          f"| val {f.val_start.date()} → {f.val_end.date()}")

EXPANDING:
  Fold 0: train 2019-07-18 → 2021-12-06 | val 2021-12-07 → 2022-07-12
    Huber loss on val set (dummy preds): 0.4191
  Fold 1: train 2019-07-18 → 2022-09-05 | val 2022-09-06 → 2023-04-11
    Huber loss on val set (dummy preds): 0.3341
  Fold 2: train 2019-07-18 → 2023-06-05 | val 2023-06-06 → 2024-01-09
    Huber loss on val set (dummy preds): 0.4042
  Fold 3: train 2019-07-18 → 2024-03-04 | val 2024-03-05 → 2024-10-08
    Huber loss on val set (dummy preds): 0.3936
  Fold 4: train 2019-07-18 → 2024-12-03 | val 2024-12-04 → 2025-07-09
    Huber loss on val set (dummy preds): 0.3501

SLIDING:
  Fold 0: train 2019-07-18 → 2021-12-06 | val 2021-12-07 → 2022-07-12
  Fold 1: train 2020-04-16 → 2022-09-05 | val 2022-09-06 → 2023-04-11
  Fold 2: train 2021-01-14 → 2023-06-05 | val 2023-06-06 → 2024-01-09
  Fold 3: train 2021-10-14 → 2024-03-04 | val 2024-03-05 → 2024-10-08
  Fold 4: train 2022-07-15 → 2024-12-03 | val 2024-12-04 → 2025-07-09


In [31]:
SEQ_LEN = 60

def model_factory(seed: int = SEED) -> GRUWithAttention:
    torch.manual_seed(seed)
    return GRUWithAttention(
    input_size=len(feature_cols), hidden_size=96,
    num_layers=3, linear_hidden=96, dropout=0.3, skip_days=10, skip_dropout=0.5
).to(DEVICE)

shared_params = dict(
    full_df=cv_df, feature_cols=feature_cols,
    model_factory=model_factory, dataset_cls=WindowDataset,
    n_folds=5, val_frac=0.10,
    seq_len=SEQ_LEN, batch_size=128, lr=1e-3,
    epochs=1000, patience=20, huber_delta=1.01, device=DEVICE,
    criterion=criterion, weight_decay=1e-1, warmup_epochs=10,
)


In [32]:
import importlib
import timeseries_cv
importlib.reload(timeseries_cv)
from timeseries_cv import cross_validate, summarize_cv, compare_strategies
exp_results = cross_validate(**shared_params, strategy="expanding")
exp_df = summarize_cv(exp_results)



  [EXPANDING] Fold 0  |  train 2019-07-18 → 2021-12-06  |  val 2021-12-07 → 2022-07-12
  Epoch   1  train_loss=0.5410  val_loss=0.4875  dir_acc=0.516
  Epoch   2  train_loss=0.5123  val_loss=0.4465  dir_acc=0.498
  Epoch   3  train_loss=0.4575  val_loss=0.4322  dir_acc=0.526
  Epoch   4  train_loss=0.4175  val_loss=0.4378  dir_acc=0.515
  Epoch   5  train_loss=0.3922  val_loss=0.4364  dir_acc=0.497
  Epoch   6  train_loss=0.3794  val_loss=0.4266  dir_acc=0.525
  Epoch   7  train_loss=0.3642  val_loss=0.4250  dir_acc=0.522
  Epoch   8  train_loss=0.3455  val_loss=0.4348  dir_acc=0.500
  Epoch   9  train_loss=0.3516  val_loss=0.4271  dir_acc=0.514
  Epoch  10  train_loss=0.3507  val_loss=0.4356  dir_acc=0.526
  Epoch  11  train_loss=0.3370  val_loss=0.4254  dir_acc=0.522
  Epoch  12  train_loss=0.3457  val_loss=0.4421  dir_acc=0.515
  Epoch  13  train_loss=0.3320  val_loss=0.4411  dir_acc=0.504
  Epoch  14  train_loss=0.3301  val_loss=0.4376  dir_acc=0.515
  Epoch  15  train_loss=0.3299

In [33]:

sli_results = cross_validate(**shared_params, strategy="sliding")
sli_df = summarize_cv(sli_results) 


  [SLIDING] Fold 0  |  train 2019-07-18 → 2021-12-06  |  val 2021-12-07 → 2022-07-12
  Epoch   1  train_loss=0.5410  val_loss=0.4875  dir_acc=0.516
  Epoch   2  train_loss=0.5123  val_loss=0.4465  dir_acc=0.498
  Epoch   3  train_loss=0.4575  val_loss=0.4322  dir_acc=0.526
  Epoch   4  train_loss=0.4175  val_loss=0.4378  dir_acc=0.515
  Epoch   5  train_loss=0.3922  val_loss=0.4364  dir_acc=0.497
  Epoch   6  train_loss=0.3794  val_loss=0.4266  dir_acc=0.525
  Epoch   7  train_loss=0.3642  val_loss=0.4250  dir_acc=0.522
  Epoch   8  train_loss=0.3455  val_loss=0.4348  dir_acc=0.500
  Epoch   9  train_loss=0.3516  val_loss=0.4271  dir_acc=0.514
  Epoch  10  train_loss=0.3507  val_loss=0.4356  dir_acc=0.526
  Epoch  11  train_loss=0.3370  val_loss=0.4254  dir_acc=0.522
  Epoch  12  train_loss=0.3457  val_loss=0.4421  dir_acc=0.515
  Epoch  13  train_loss=0.3320  val_loss=0.4411  dir_acc=0.504
  Epoch  14  train_loss=0.3301  val_loss=0.4376  dir_acc=0.515
  Epoch  15  train_loss=0.3299  

In [34]:
comp_df = compare_strategies(exp_results, sli_results)


  STRATEGY COMPARISON
  expanding   |  dir_acc = 0.549 ± 0.015  |  val_loss = 0.3574 ± 0.0432
  sliding     |  dir_acc = 0.541 ± 0.021  |  val_loss = 0.3620 ± 0.0416

  → Expanding wins: model benefits from more history.
  → Expanding is more stable across folds (lower variance).


In [35]:
ENSEMBLE_SEEDS = [42, 1337, 7777, 2024]
import importlib
import timeseries_cv
importlib.reload(timeseries_cv)
from timeseries_cv import ensemble_cross_validate


ens_cv = ensemble_cross_validate(
    seeds=ENSEMBLE_SEEDS,
    make_model=model_factory,
    full_df=cv_df,
    feature_cols=feature_cols,
    dataset_cls=WindowDataset,
    strategy="expanding",
    n_folds=5,
    val_frac=0.10,
    seq_len=SEQ_LEN,
    batch_size=128,
    lr=1e-3,
    epochs=1000,
    patience=20,
    huber_delta=1.01,
    device=DEVICE,
    criterion=criterion,
    weight_decay=1e-1,
    warmup_epochs=10,
)


############################################################
  ENSEMBLE SEED 1/4  seed=42
############################################################

  [EXPANDING] Fold 0  |  train 2019-07-18 → 2021-12-06  |  val 2021-12-07 → 2022-07-12
  Epoch   1  train_loss=0.5411  val_loss=0.5481  dir_acc=0.492
  Epoch   2  train_loss=0.5000  val_loss=0.4926  dir_acc=0.481
  Epoch   3  train_loss=0.4586  val_loss=0.4607  dir_acc=0.483
  Epoch   4  train_loss=0.4093  val_loss=0.4330  dir_acc=0.519
  Epoch   5  train_loss=0.3778  val_loss=0.4289  dir_acc=0.520
  Epoch   6  train_loss=0.3647  val_loss=0.4378  dir_acc=0.483
  Epoch   7  train_loss=0.3548  val_loss=0.4167  dir_acc=0.532
  Epoch   8  train_loss=0.3473  val_loss=0.4293  dir_acc=0.533
  Epoch   9  train_loss=0.3496  val_loss=0.4458  dir_acc=0.482
  Epoch  10  train_loss=0.3420  val_loss=0.4301  dir_acc=0.517
  Epoch  11  train_loss=0.3392  val_loss=0.4387  dir_acc=0.532
  Epoch  12  train_loss=0.3307  val_loss=0.4557  dir_acc=0.487
  Ep

KeyboardInterrupt: 

In [ ]:
import plotly.graph_objects as go

fold_labels = [f"Fold {r.fold_idx}" for r in exp_results]

fig = go.Figure()
fig.add_trace(go.Bar(
    x=fold_labels,
    y=[r.val_dir_acc for r in exp_results],
    name="Expanding", marker_color="#636efa", opacity=0.8,
))
# fig.add_trace(go.Bar(
#     x=fold_labels,
#     y=[r.val_dir_acc for r in sli_results],
#     name="Sliding", marker_color="#ef553b", opacity=0.8,
# ))
fig.add_hline(y=0.50, line_dash="dash", line_color="white",
              annotation_text="coin flip")
fig.update_layout(
    template="plotly_dark",
    title="Expanding vs Sliding Window — Dir. Accuracy per Fold",
    yaxis_title="Dir. Accuracy",
    yaxis_range=[0.40, 0.65],
    barmode="group",
)
fig.show()

In [ ]:
from timeseries_cv import evaluate_holdout
btc_result = evaluate_holdout(
    train_df=cv_df, eval_df=test_df,
    feature_cols=feature_cols,
    model_factory=model_factory,
    dataset_cls=WindowDataset,
    lr=1e-3, device=DEVICE,
)



  HOLDOUT EVALUATION  |  ticker=ALL
  train 2019-07-18 -> 2025-07-09  (12,011 rows)
  eval  2025-07-10 -> 2026-03-10  (1,464 rows)
  Epoch   1  train_loss=0.5104  eval_loss=0.4754  dir_acc=0.490
  Epoch   2  train_loss=0.4044  eval_loss=0.4276  dir_acc=0.497
  Epoch   3  train_loss=0.3738  eval_loss=0.4279  dir_acc=0.519
  Epoch   4  train_loss=0.3720  eval_loss=0.4275  dir_acc=0.506
  Epoch   5  train_loss=0.3711  eval_loss=0.4362  dir_acc=0.485
  Epoch   6  train_loss=0.3694  eval_loss=0.4317  dir_acc=0.493
  Epoch   7  train_loss=0.3678  eval_loss=0.4338  dir_acc=0.516
  Epoch   8  train_loss=0.3655  eval_loss=0.4351  dir_acc=0.511
  Epoch   9  train_loss=0.3649  eval_loss=0.4358  dir_acc=0.494
  Epoch  10  train_loss=0.3627  eval_loss=0.4450  dir_acc=0.496
  Epoch  11  train_loss=0.3604  eval_loss=0.4342  dir_acc=0.497
  Epoch  12  train_loss=0.3577  eval_loss=0.4432  dir_acc=0.491
  Epoch  13  train_loss=0.3580  eval_loss=0.4375  dir_acc=0.484
  Epoch  14  train_loss=0.3561  eval

In [ ]:
import importlib
import timeseries_cv
importlib.reload(timeseries_cv)
from timeseries_cv import evaluate_holdout, EvalResult


In [ ]:
btc_result = evaluate_holdout(
    train_df=cv_df, eval_df=test_df,
    feature_cols=feature_cols,
    model_factory=model_factory,
    dataset_cls=WindowDataset,
    lr=1e-3, device=DEVICE,
    ticker=["BTC-USD", "ETH-USD"],
)



  HOLDOUT EVALUATION  |  ticker=BTC-USD, ETH-USD
  train 2019-07-18 -> 2025-07-09  (4,368 rows)
  eval  2025-07-10 -> 2026-03-10  (488 rows)
  Epoch   1  train_loss=0.5335  eval_loss=0.5354  dir_acc=0.500
  Epoch   2  train_loss=0.4466  eval_loss=0.4580  dir_acc=0.491
  Epoch   3  train_loss=0.3907  eval_loss=0.4547  dir_acc=0.520
  Epoch   4  train_loss=0.3728  eval_loss=0.4524  dir_acc=0.529
  Epoch   5  train_loss=0.3695  eval_loss=0.4598  dir_acc=0.471
  Epoch   6  train_loss=0.3719  eval_loss=0.4518  dir_acc=0.509
  Epoch   7  train_loss=0.3701  eval_loss=0.4537  dir_acc=0.525
  Epoch   8  train_loss=0.3689  eval_loss=0.4545  dir_acc=0.516
  Epoch   9  train_loss=0.3668  eval_loss=0.4613  dir_acc=0.498
  Epoch  10  train_loss=0.3687  eval_loss=0.4591  dir_acc=0.496
  Epoch  11  train_loss=0.3633  eval_loss=0.4654  dir_acc=0.498
  Epoch  12  train_loss=0.3618  eval_loss=0.4660  dir_acc=0.500
  Epoch  13  train_loss=0.3612  eval_loss=0.4742  dir_acc=0.489
  Epoch  14  train_loss=0.

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=btc_result.eval_dates, y=btc_result.eval_targets, name="actual"))
fig.add_trace(go.Scatter(x=btc_result.eval_dates, y=btc_result.eval_preds,   name="predicted"))
fig.show()